# Inferencia reproducible: del CSV a la predicción

Notebook docente de la clase 1. Se ejecuta con un modelo ya entrenado; no se entrena ningún modelo aquí.

**Secuencia:** contrato → validación → preprocesado → inferencia → salida trazable.

## Preparación

Desde la raíz del repositorio:

```bash
uv sync
```

El profesorado debe tener `models/wine_quality_classifier.joblib`. Las celdas de inferencia comprueban si el artefacto está disponible.

In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path
import csv
import subprocess
import sys

import joblib
from pydantic import BaseModel, ConfigDict, Field, ValidationError

from model_inference.contracts import (
    WineQualityPrediction,
    WineQualityRequest as ReferenceWineQualityRequest,
)
from model_inference.inference import (
    DEFAULT_MODEL_PATH,
    infer_wine_quality,
    load_wine_quality_model,
)
from model_inference.preprocess import (
    FEATURE_NAMES,
    PREPROCESSING_VERSION,
    preprocess_wine_request,
)

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "assets/03-wine-quality/inference_samples.csv").is_file():
        ROOT = candidate
        break

DATA_PATH = ROOT / "assets/03-wine-quality/inference_samples.csv"
MODEL_PATH = ROOT / DEFAULT_MODEL_PATH
print(f"Repositorio: {ROOT}")
print(f"Datos: {DATA_PATH.exists()} | Modelo: {MODEL_PATH.exists()}")

## 1. Mirar los datos nuevos

Antes de hablar de modelos, identifica qué información llega en una fila.

In [ ]:
with DATA_PATH.open(newline="", encoding="utf-8") as source:
    rows = list(csv.DictReader(source))

print(f"Muestras: {len(rows)}")
print(f"Columnas: {list(rows[0])}")
rows[:2]

### Práctica 1 — Separar piezas

En parejas, responded sin ejecutar más código:

- ¿Qué dato identifica la petición?
- ¿Qué 11 campos consume el modelo?
- ¿Qué información se recibió ya hecha?
- ¿Qué columna añadiríais para provocar un error de contrato?

## 2. El mismo contrato con Pydantic

Pydantic valida la frontera de entrada: campos, tipos, rangos y columnas extra. `sample_id` acompaña a la petición, pero no entra en el vector del modelo.

In [ ]:
class WineQualityRequestPydantic(BaseModel):
    model_config = ConfigDict(extra="forbid")

    fixed_acidity: float = Field(ge=0, le=20)
    volatile_acidity: float = Field(ge=0, le=2)
    citric_acid: float = Field(ge=0, le=2)
    residual_sugar: float = Field(ge=0, le=20)
    chlorides: float = Field(ge=0, le=1)
    free_sulfur_dioxide: float = Field(ge=0, le=100)
    total_sulfur_dioxide: float = Field(ge=0, le=300)
    density: float = Field(ge=0.98, le=1.01)
    ph: float = Field(ge=2.5, le=4.5)
    sulphates: float = Field(ge=0, le=3)
    alcohol: float = Field(ge=5, le=20)

row = rows[0]
sample_id = row["sample_id"]
model_row = {name: row[name] for name in FEATURE_NAMES}
request = WineQualityRequestPydantic.model_validate(model_row)
print(f"sample_id: {sample_id}")
print(request)

### Práctica 2 — Provocar errores de entrada

Antes de ejecutar la siguiente celda, predice qué ocurrirá con una columna desconocida y con un `ph` imposible.

In [ ]:
cases = {
    "columna extra": {**model_row, "unexpected_field": "not allowed"},
    "rango inválido": {**model_row, "ph": "99"},
}

for name, candidate in cases.items():
    try:
        WineQualityRequestPydantic.model_validate(candidate)
    except ValidationError as error:
        print(f"{name}: rechazado → {error.errors()[0]['msg']}")

## 3. El mismo contrato con `dataclass`

Una `dataclass` expresa la estructura, pero no valida automáticamente tipos, rangos ni campos extra. La validación hay que programarla o delegarla en otra capa.

In [ ]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    fixed_acidity: float
    volatile_acidity: float
    citric_acid: float
    residual_sugar: float
    chlorides: float
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float
    density: float
    ph: float
    sulphates: float
    alcohol: float

dataclass_request = WineQualityRequestDataclass(**request.model_dump())
print(asdict(dataclass_request))

# La anotación float no convierte ni valida por sí sola.
unvalidated = WineQualityRequestDataclass(**{**request.model_dump(), "ph": 99.0})
print(f"dataclass acepta ph=99.0: {unvalidated.ph}")

### Práctica 3 — Decisión de diseño

En parejas, completad esta frase:

> En la frontera de entrada usamos Pydantic porque __________; usamos `dataclass` para __________.

## 4. Preprocesado: construir el vector

El modelo no recibe el diccionario ni el CSV. Recibe una lista numérica en un orden estable.

In [ ]:
reference_request = ReferenceWineQualityRequest.model_validate(model_row)
features = preprocess_wine_request(reference_request)
vector = features.as_vector()

for position, (name, value) in enumerate(zip(FEATURE_NAMES, vector, strict=True)):
    print(f"{position:02d}  {name:24s} {value}")

### Práctica 4 — Detectar el error de orden

Intercambiad mentalmente `density` y `alcohol`. ¿El vector sigue teniendo 11 valores? ¿Representa la misma muestra? Explicad por qué el modelo podría no lanzar ningún error.

## 5. Cargar el artefacto e inferir

El cargador comprueba que el artefacto declara los mismos `feature_names` que el preprocesado.

In [ ]:
if not MODEL_PATH.is_file():
    print("Falta el artefacto docente en models/wine_quality_classifier.joblib")
else:
    payload = joblib.load(MODEL_PATH)
    print(f"Claves del artefacto: {list(payload)}")
    print(f"Orden compatible: {tuple(payload['feature_names']) == FEATURE_NAMES}")
    print(f"Versión: {payload['model_version']}")

In [ ]:
if MODEL_PATH.is_file():
    loaded_model = load_wine_quality_model(MODEL_PATH)
    quality_band, confidence = infer_wine_quality(loaded_model, features)
    prediction = WineQualityPrediction(
        quality_band=quality_band,
        confidence=confidence,
        model_version=loaded_model.model_version,
        preprocessing_version=PREPROCESSING_VERSION,
    )
    output_row = {"sample_id": sample_id, **prediction.model_dump()}
    output_row
else:
    print("Coloca el modelo docente para ejecutar esta celda.")

### Práctica 5 — Leer la predicción

Interpretad `quality_band`, `confidence`, `model_version` y `preprocessing_version`. La confianza es la probabilidad máxima del clasificador; no es una garantía de calidad.

## 6. Demo completa por CLI

Ahora el mismo recorrido se encapsula en un comando reutilizable. El alumnado observa el comportamiento; el módulo completo se implementa en la clase 2.

In [ ]:
output_path = ROOT / ".tmp/wine_predictions_from_notebook.csv"
output_path.parent.mkdir(exist_ok=True)

if MODEL_PATH.is_file():
    completed = subprocess.run(
        [
            sys.executable,
            "-m",
            "model_inference.predict_file",
            "--input",
            str(DATA_PATH),
            "--output",
            str(output_path),
        ],
        cwd=ROOT,
        capture_output=True,
        text=True,
    )
    print(f"returncode={completed.returncode}")
    print(completed.stdout or completed.stderr)
    print(output_path.read_text(encoding="utf-8"))
else:
    print("Coloca el modelo docente para ejecutar la demo CLI.")

### Práctica 6 — Romper el contrato sin tocar el código

Añade una columna `unexpected_field` a una copia temporal del CSV y ejecuta de nuevo el comando. Predice primero el resultado: debe fallar y no generar un CSV de salida.

In [ ]:
invalid_input = ROOT / ".tmp/inference_samples_invalid.csv"
invalid_output = ROOT / ".tmp/invalid_predictions.csv"
invalid_input.parent.mkdir(exist_ok=True)

with DATA_PATH.open(newline="", encoding="utf-8") as source:
    original_rows = list(csv.DictReader(source))
invalid_output.unlink(missing_ok=True)
fieldnames = ["sample_id", *FEATURE_NAMES, "unexpected_field"]
with invalid_input.open("w", newline="", encoding="utf-8") as target:
    writer = csv.DictWriter(target, fieldnames=fieldnames)
    writer.writeheader()
    for original_row in original_rows:
        writer.writerow({**original_row, "unexpected_field": "not allowed"})

if MODEL_PATH.is_file():
    completed = subprocess.run(
        [
            sys.executable, "-m", "model_inference.predict_file",
            "--input", str(invalid_input),
            "--output", str(invalid_output),
        ],
        cwd=ROOT, capture_output=True, text=True,
    )
    print(f"returncode={completed.returncode}")
    print(completed.stderr)
    print(f"salida creada: {invalid_output.exists()}")
else:
    print("Coloca el modelo docente para ejecutar la demo de error.")

## Cierre

Antes de la clase 2, cada pareja debe poder explicar:

1. Qué valida el contrato.
2. Cómo se construye el vector.
3. Qué comprueba el cargador del artefacto.
4. Qué columnas tiene la salida.

En la clase 2 implementarán `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py` sobre el starter.